In [1]:
from google.colab import files
uploaded = files.upload()

Saving BanglaPSG-Grievance-Dataset_V2.xlsx to BanglaPSG-Grievance-Dataset_V2.xlsx


In [4]:
!pip install pandas openpyxl scikit-learn -q

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [6]:
df = pd.read_excel("BanglaPSG-Grievance-Dataset_V2.xlsx")

In [7]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (6105, 4)

Columns:
['No.', 'Grievance Text', 'Category', 'Severity']


In [8]:
# First 5 rows
df.head()

,No.,Grievance Text,Category,Severity
0,1,সরকারি হাসপাতাল গুলো এত অপরিষ্কার আর নোংরা।,Healthcare,High
1,2,মেডিসিন বিশেষজ্ঞ ডাঃ বসে না এসিস্ট্যান্ট দিয়ে ...,Healthcare,High
2,3,রূপগঞ্জে মানুষ মারার কারখানা ইউ এস-বাংলা হাসপাতাল,Healthcare,Low
3,4,সরকারি হাসপাতালগুলোর সমস্ত টেস্টের মেশিন গুলো ...,Healthcare,Medium
4,5,সামান্য জ্বর আসলে এক হাজার নয়শত টাকার টেস্ট দয়,Healthcare,Low


In [9]:
# Dataset Information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6105 entries, 0 to 6104
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   No.             6105 non-null   int64 
 1   Grievance Text  6105 non-null   object
 2   Category        6105 non-null   object
 3   Severity        6105 non-null   object
dtypes: int64(1), object(3)
memory usage: 190.9+ KB


In [10]:
# Missing Values
print(df.isnull().sum())

No.               0
Grievance Text    0
Category          0
Severity          0
dtype: int64


In [11]:
# Category Distribution
print(df["Category"].value_counts())

Category
Road & Transport    1061
Food                1036
Waste               1015
Healthcare          1015
Electricity         1007
Water                971
Name: count, dtype: int64


In [12]:
print(df["Severity"].value_counts())

Severity
Medium    2464
Low       1988
High      1653
Name: count, dtype: int64


In [14]:
X = df["Grievance Text"]
y = df["Category"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

y_val_pred = svm.predict(X_val_tfidf)
y_pred = svm.predict(X_test_tfidf)

print("\n-- Validation Results --")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, average='weighted'))
print("Recall:", recall_score(y_val, y_val_pred, average='weighted'))
print("F1-score:", f1_score(y_val, y_val_pred, average='weighted'))

print("\n -- Test Results (Final) --")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1-score:", f1_score(y_test, y_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Train size: 4273
Validation size: 916
Test size: 916

-- Validation Results --
Accuracy: 0.7194323144104804
Precision: 0.7218738263492255
Recall: 0.7194323144104804
F1-score: 0.720338145077272

 -- Test Results (Final) --
Accuracy: 0.6932314410480349
Precision: 0.6953931076274006
Recall: 0.6932314410480349
F1-score: 0.6939655497728138

Classification Report:
                   precision    recall  f1-score   support

     Electricity       0.73      0.67      0.70       151
            Food       0.66      0.65      0.65       156
      Healthcare       0.78      0.78      0.78       152
Road & Transport       0.70      0.70      0.70       159
           Waste       0.76      0.76      0.76       153
           Water       0.55      0.59      0.57       145

        accuracy                           0.69       916
       macro avg       0.69      0.69      0.69       916
    weighted avg       0.70      0.69      0.69       916

